In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_7_try_0.pickle

In [ ]:
%%PandasProfile
### cell 8 ###
def combine_subset_of_data_from_multiple_years_20(question_of_interest, x_axis_title, include_2017=None):
    # Define year-dataframe pairs in desired order
    pairs = [
        ('2018', responses_df_2018_cell10),
        ('2019', responses_df_2019_cell10),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022_cell10),
    ]
    if include_2017 is not None:
        pairs.insert(0, ('2017', responses_df_2017))
    # Precompute non-null counts per year
    denom = {year: df[question_of_interest].count() for year, df in pairs}
    # Concatenate all years into one DataFrame and rename column for x-axis
    combined = pd.concat(
        [df[[question_of_interest]].assign(year=year) for year, df in pairs],
        ignore_index=True
    ).rename(columns={question_of_interest: x_axis_title})
    # Compute counts (including NaNs) and then percentages
    counts = combined.groupby(['year', x_axis_title], dropna=False).size()
    percentages = (counts * 100).div(counts.index.get_level_values('year').map(denom)).round(1)
    # Build final DataFrame with desired column order
    return (
        percentages
        .reset_index(name='percentage')
        [['percentage', 'year', x_axis_title]]
    )

# Parameters and preprocessing
question_of_interest_cell20 = 'What is your age (# years)?'
title_for_x_axis_cell20 = ''
# Merge upper age buckets before computing
responses_df_2018_cell10[question_of_interest_cell20].replace(['70-79', '80+'], '70+', inplace=True)

# Combine and inspect
age_df_combined = combine_subset_of_data_from_multiple_years_20(
    question_of_interest_cell20,
    title_for_x_axis_cell20
)
age_df_combined.info()